In [15]:
from timeit import default_timer as timer
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from IPython.display import HTML
import warnings

warnings.filterwarnings("ignore")

In [16]:
x, y = fetch_openml(name="spoken-arabic-digit", return_X_y=True)

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=123)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((236930, 14), (26326, 14), (236930,), (26326,))

In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler_x = MinMaxScaler()

In [19]:
scaler_x.fit(x_train)
x_train = scaler_x.transform(x_train)
x_test = scaler_x.transform(x_test)

In [20]:

from sklearnex import patch_sklearn

patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [21]:
from sklearn.cluster import KMeans

params = {
    "n_clusters": 128,
    "random_state": 123,
    "copy_x": False,
}
start = timer()
model = KMeans(**params).fit(x_train, y_train)
train_patched = timer() - start
f"Extension for Scikit-learn time: {train_patched:.2f} s"

'Extension for Scikit-learn time: 0.94 s'

In [22]:

inertia_opt = model.inertia_
n_iter_opt = model.n_iter_
print(f"Extension for Scikit-learn inertia: {inertia_opt}")
print(f"Extension for Scikit-learn number of iterations: {n_iter_opt}")

Extension for Scikit-learn inertia: 13381.464513943798
Extension for Scikit-learn number of iterations: 198


In [23]:
from sklearnex import unpatch_sklearn

unpatch_sklearn()

In [24]:
from sklearn.cluster import KMeans

start = timer()
model = KMeans(**params).fit(x_train, y_train)
train_unpatched = timer() - start
f"Original Scikit-learn time: {train_unpatched:.2f} s"

'Original Scikit-learn time: 1.98 s'

In [25]:

inertia_original = model.inertia_
n_iter_original = model.n_iter_
print(f"Original Scikit-learn inertia: {inertia_original}")
print(f"Original Scikit-learn number of iterations: {n_iter_original}")

Original Scikit-learn inertia: 13377.716442347584
Original Scikit-learn number of iterations: 244


In [26]:
HTML(
    f"<h3>Compare inertia and number of iterations of patched Scikit-learn and original</h3><br>"
    f"<strong>Inertia:</strong><br>"
    f"Patched Scikit-learn: {inertia_opt} <br>"
    f"Unpatched Scikit-learn: {inertia_original} <br>"
    f"Ratio: {inertia_opt/inertia_original} <br><br>"
    f"<strong>Number of iterations:</strong><br>"
    f"Patched Scikit-learn: {n_iter_opt} <br>"
    f"Unpatched Scikit-learn: {n_iter_original} <br>"
    f"Ratio: {(n_iter_opt/n_iter_original):.2f} <br><br>"
    f"Number of iterations is bigger but algorithm is much faster and inertia is lower"
    f"<h3>With Scikit-learn-intelex patching you can:</h3>"
    f"<ul>"
    f"<li>Use your Scikit-learn code for training and prediction with minimal changes (a couple of lines of code);</li>"
    f"<li>Fast execution training and prediction of Scikit-learn models;</li>"
    f"<li>Get speedup in <strong>{(train_unpatched/train_patched):.1f}</strong> times.</li>"
    f"</ul>"
)